# Churn Model Comparison

A second notebook containing data preparation, exploratory analysis, and model comparisons. It reads the same local dataset as `churn_rate_exploration.ipynb`.

**TASK-1  , DATA PREPERATION**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# Load the local dataset
from pathlib import Path

DATA_PATH = Path("data/telco_customer_churn.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError("Place the Telco Customer Churn CSV at data/telco_customer_churn.csv.")
df = pd.read_csv(DATA_PATH)


In [ ]:
df.head(4)

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
nulls_in_columns = df.isnull().sum()
print("Null values in each column:")
print(nulls_in_columns)

In [ ]:
nulls_in_rows = df.isnull().sum(axis=1)
print("Null values in each row:")
print(nulls_in_rows)

In [ ]:
mean=df["TotalCharges"].mean()
df["TotalCharges"].replace(np.NaN,mean,inplace=True)

In [ ]:
nulls_in_columns = df.isnull().sum()
print("Null values in each column:")
print(nulls_in_columns)

In [ ]:
nulls_in_rows = df.isnull().sum(axis=1)
print("Null values in each row:")
print(nulls_in_rows)

In [ ]:
# Categorical variable encoding
categorical_columns = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService',
                       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
                       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

df_encoded = pd.get_dummies(df, columns=categorical_columns)
df_encoded

In [ ]:
for column in df.columns:
  unique_values=df[column].unique()
  print(f'uique value in {column}: ',unique_values)

In [ ]:
# Dataset splitting.
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

**TASK-2 , Exploratory Data Analysis (EDA)**

In [ ]:
df.columns

In [ ]:
fig,ax=plt.subplots()
sns.histplot(df['tenure'],kde=True)
plt.title('distribution of tenure')

In [ ]:
fig,axs=plt.subplots(1,2, figsize=(17,5))

axs[0].hist(df['Contract'])
axs[0].set_title('contract type distribution')

axs[1].hist(df["PaymentMethod"])
axs[1].set_title('payment method distribution')

In [ ]:
sns.countplot(x='Churn',data=df)
plt.title('churn distribution')

In [ ]:
mean_monthly_charges = df_encoded.groupby('Churn')['MonthlyCharges'].mean()
plt.figure(figsize=(8, 6))
plt.bar(mean_monthly_charges.index, mean_monthly_charges.values, color=['green', 'red'])
plt.xlabel('Churn Status')
plt.ylabel('Mean Monthly Charges')
plt.title('Mean Monthly Charges by Churn Status')
plt.xticks(mean_monthly_charges.index, ['Not Churned', 'Churned'])
plt.show()


In [ ]:
total_charges_by_churn = df_encoded.groupby('Churn')['TotalCharges'].mean()
plt.figure(figsize=(8, 6))
plt.bar(total_charges_by_churn.index, total_charges_by_churn.values, color=['green', 'red'])
plt.xlabel('Churn Status')
plt.ylabel('Total Charges')
plt.title('Total Charges by Churn Status')
plt.xticks(total_charges_by_churn.index, ['Not Churned', 'Churned'])
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
sns.kdeplot(data=df_encoded, x='MonthlyCharges', hue='Churn', shade=True)
plt.xlabel('Total Charges')
plt.ylabel('Density')
plt.title('Kernel Density Estimation of Total Charges by Churn Status')
plt.legend(title='Churn Status', labels=['Not Churned', 'Churned'])
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.kdeplot(data=df_encoded, x='TotalCharges', hue='Churn', shade=True)
plt.xlabel('Total Charges')
plt.ylabel('Density')
plt.title('Kernel Density Estimation of Total Charges by Churn Status')
plt.legend(title='Churn Status', labels=['Not Churned', 'Churned'])
plt.show()

In [ ]:
df_encoded['Churn_numeric'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})
pearson_coef, p_value = pearsonr(df_encoded['TotalCharges'], df_encoded['Churn_numeric'])
print("Pearson correlation coefficient:", pearson_coef)
print("P-value:", p_value)

In [ ]:
df_encoded['Churn_numeric'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})
pearson_coef, p_value = pearsonr(df_encoded['MonthlyCharges'], df_encoded['Churn_numeric'])
print("Pearson correlation coefficient:", pearson_coef)
print("P-value:", p_value)

In [ ]:
for param in ['MonthlyCharges', 'TotalCharges']:
    df_encoded['Churn_numeric'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})
    print(f'Correlation of Churn and {param}:')
    print(df_encoded[[param, 'Churn_numeric']].corr())
    df_encoded.drop(columns=['Churn_numeric'], inplace=True)

In [ ]:
for param in ['MonthlyCharges', 'TotalCharges']:
    df_encoded['Churn_numeric'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})
    correlation = df_encoded['Churn_numeric'].corr(df_encoded[param])
    print(f'Correlation of Churn and {param}: {correlation}')
    df_encoded.drop(columns=['Churn_numeric'], inplace=True)


In [ ]:
df_encoded['Churn'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})
num_churned_customers = df_encoded['Churn'].sum()
total_customers = len(df_encoded)
churn_rate = (num_churned_customers / total_customers) * 100
print("Churn Rate:", churn_rate)

**Customer Segmentation**



In [ ]:
df.columns

In [ ]:
df_encoded.head(2)

In [ ]:
df.head(2)

In [ ]:
df["Churn_Yes_Num"]=df["Churn"].map({"Yes":1,"No":0})
df

In [ ]:
sum_variable=df["Churn_Yes_Num"].sum()
sum_variable

In [ ]:
def segment_tenure(tenure):
  if tenure < 1:
    return "new costumer"
  elif tenure>=1 and tenure<=1:
    return "mid term customer"
  else:
    return "long term costumer"

df["tenure_segment"]=df['tenure'].apply(segment_tenure)

def segment_charges(charges):
  if charges < 50:
    return "low charge"
  elif charges>=50 and charges<=100:
    return "mid charge"
  else:
    return "high charge"

df["charges_segment"]=df['MonthlyCharges'].apply(segment_charges)


def segment_contract_type(contract_type):
  if contract_type == 'Month-to-Month':
    return "month to month contract"
  elif contract_type == 'One year':
    return "one year contract"
  else:
    return "two year contract"

df["contract_segment"]=df['Contract'].apply(segment_contract_type)

df[["tenure_segment","charges_segment","contract_segment"]]
df





In [ ]:
def calculate_churn_rate(segment):
  total_customers=len(segment)
  churned_customers=segment['Churn_Yes_Num'].sum()
  churn_rate = (churned_customers / total_customers)*100
  return churn_rate

tenure_segment=df.groupby('tenure_segment')
tenure_churn_rate=tenure_segment.apply(calculate_churn_rate)

In [ ]:
def calculate_churn_rate(segment):
    total_customers = len(segment)
    churned_customers = segment['Churn'].sum()
    churn_rate = (churned_customers / total_customers) * 100
    return churn_rate

# Segment customers based on tenure
tenure_segment = df.groupby('tenure_segment')
tenure_churn_rate = tenure_segment.apply(calculate_churn_rate)

# Segment customers based on monthly charges
charges_segment = df.groupby('charges_segment')
charges_churn_rate = charges_segment.apply(calculate_churn_rate)

# Segment customers based on contract type
contract_segment = df.groupby('contract_segment')
contract_churn_rate = contract_segment.apply(calculate_churn_rate)

In [ ]:
# Segment customers based on tenure
customer_segments = df.groupby('tenure_segment')
segment_churn_rates = customer_segments.apply(calculate_churn_rate)

In [ ]:
def calculate_churn_rates(df):
    segments = define_segments(df)
    churn_rates = calculate_rates(segments)
    print_rates(churn_rates)

In [ ]:
def calculate_churn_rate(segment):
    total_customers = len(segment)
    churned_customers = segment['Churn_Yes'].sum()
    churn_rate = (churned_customers / total_customers) * 100
    return churn_rate

def calculate_churn_rates(df):
    segments = define_segments(df)
    churn_rates = calculate_rates(segments)
    print_rates(churn_rates)

def define_segments(df):
    return df.groupby(['tenure_segment', 'charges_segment', 'contract_segment'])

def calculate_rates(segments):
    return segments.apply(calculate_churn_rate)

def print_rates(churn_rates):
    print(churn_rates)


In [ ]:
churn_rates = {
    'tenure': segment_churn_rates,
    'charges': charges_churn_rate,
    'contract': contract_churn_rate
}

print(churn_rates)


In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
selected_columns = ['tenure', 'Churn']
X = df[selected_columns]
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=0)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.plot(range(1, 11), wcss)
plt.title('Elbow Method')
plt.xlabel('churn')
plt.ylabel('tenure')
plt.show()


In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
selected_columns = ['MonthlyCharges', 'Churn']
X = df[selected_columns]
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=0)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.plot(range(1, 11), wcss)
plt.title('Elbow Method')
plt.xlabel('churn')
plt.ylabel('monthly charges')
plt.show()


In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
selected_columns = ['Contract_Month-to-month', 'Churn']
X = df_encoded[selected_columns]
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=0)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.plot(range(1, 11), wcss)
plt.title('Elbow Method')
plt.xlabel('churn')
plt.ylabel('contract_month to month')
plt.show()


In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
selected_columns = ['Contract_One year', 'Churn']
X = df_encoded[selected_columns]
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=0)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.plot(range(1, 11), wcss)
plt.title('Elbow Method')
plt.xlabel('churn')
plt.ylabel('contract one year')
plt.show()


In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
selected_columns = ['Contract_Two year', 'Churn']
X = df_encoded[selected_columns]
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=0)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.plot(range(1, 11), wcss)
plt.title('Elbow Method')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.show()


In [ ]:
df_encoded.head(2)

In [ ]:
features = ['tenure', 'Churn']

X = StandardScaler().fit_transform(df_encoded[features])

kmeans = KMeans(n_clusters=3, random_state=0)
df_encoded['Cluster'] = kmeans.fit_predict(X)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_encoded, x='Churn', y='tenure', hue='Cluster', palette='viridis')
plt.title('Customer Segmentation')
plt.xlabel('churn')
plt.ylabel('tenure')
plt.legend(title='Churn Status', labels=['cluster 0', 'cluster 1','cluster 2','cluster 3'])
plt.show()

In [ ]:
features = ['Contract_Month-to-month', 'Churn']

X = StandardScaler().fit_transform(df_encoded[features])

kmeans = KMeans(n_clusters=3, random_state=0)
df_encoded['Cluster'] = kmeans.fit_predict(X)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_encoded, x='Churn', y='Contract_Month-to-month', hue='Cluster', palette='viridis')
plt.title('Customer Segmentation')
plt.xlabel('churn')
plt.ylabel('Contract_Month-to-month')
plt.legend(title='Churn Status', labels=['cluster 0', 'cluster 1','cluster 2','cluster 3'])
plt.show()

In [ ]:
features = ['Contract_One year', 'Churn']

X = StandardScaler().fit_transform(df_encoded[features])

kmeans = KMeans(n_clusters=3, random_state=0)
df_encoded['Cluster'] = kmeans.fit_predict(X)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_encoded, x='Churn', y='Contract_One year', hue='Cluster', palette='viridis')
plt.title('Customer Segmentation')
plt.xlabel('churn')
plt.ylabel('Contract_One year')
plt.legend(title='Churn Status', labels=['cluster 0', 'cluster 1','cluster 2','cluster 3'])
plt.show()

In [ ]:
features = ['Contract_Two year', 'Churn']

X = StandardScaler().fit_transform(df_encoded[features])

kmeans = KMeans(n_clusters=3, random_state=0)
df_encoded['Cluster'] = kmeans.fit_predict(X)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_encoded, x='Churn', y='Contract_Two year', hue='Cluster', palette='viridis')
plt.title('Customer Segmentation')
plt.xlabel('churn')
plt.ylabel('Contract_Two year')
plt.legend(title='Churn Status', labels=['cluster 0', 'cluster 1','cluster 2','cluster 3'])
plt.show()

In [ ]:
df_encoded.head(2)

In [ ]:
features = ['MonthlyCharges', 'Churn']

X = StandardScaler().fit_transform(df_encoded[features])

kmeans = KMeans(n_clusters=3, random_state=0)
df_encoded['Cluster'] = kmeans.fit_predict(X)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_encoded, x='Churn', y='MonthlyCharges', hue='Cluster', palette='viridis')
plt.title('Customer Segmentation')
plt.xlabel('churn')
plt.ylabel('Monthly Charges')
plt.legend(title='Churn Status', labels=['cluster 0', 'cluster 1','cluster 2','cluster 3'])
plt.show()

**Task 4: Churn Prediction Model**

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
lr = LinearRegression()
X = df[['MonthlyCharges', 'TotalCharges']]
Y = df_encoded['Churn']
lr.fit(X,Y)
Y_hat = lr.predict(X)
coeff = lr.coef_
intercept = lr.intercept_
print(coeff)
print(intercept)

In [ ]:
df.columns

In [ ]:
df.head(2)

In [ ]:
df_encoded.head(2)

In [ ]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import GridSearchCV

# Load data
data = pd.read_csv(DATA_PATH)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data.drop("Churn", axis=1), data["Churn"], test_size=0.2, random_state=42)

# Train multiple models
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"Model: {name}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
    print(f"Precision: {precision_score(y_test, y_pred)}")
    print(f"Recall: {recall_score(y_test, y_pred)}")
    print(f"F1 Score: {f1_score(y_test, y_pred)}")
    print()

# Perform feature selection
selector = SelectKBest(f_regression, k=5)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Train models again with selected features
for name, model in models.items():
    model.fit(X_train_selected, y_train)
    y_pred = model.predict(X_test_selected)
    print(f"Model: {name} (with feature selection)")
    print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
    print(f"Precision: {precision_score(y_test, y_pred)}")

Bref about all the task and all

Task 1 [Data Preparation] : performed various operation on the data set like finding missing value replacing it with mean or removing the column splitting the data set so that further model be prepared for prediction ,converting categorical variable encoding for non numeric entry gain necessary understanding of the data set and clean it to get basic insite is of the data

skills used :
Data loading, data exploration,
Handling missing values,
Data preprocessing,
Categorical variable encoding,
Dataset splitting.

Task 2 [Exploratory Data Analysis (EDA)] :  performed various operation to gain visual insite of the data "how" they are related to each other what are the factor that affect the churn rate ploted different plots like 'bar graph' , 'Kernel Density Estimation plot' ;
calculation of pearsons correlation  p value and  correlation by how much number they are affecting the churn  affter performing
basic assumption /prediction : churn rate is mostly affected by monthly charge and total charge

Important statical finding

total charge vs churn rate
Pearson correlation coefficient: -0.1994277150883139
P-value: 4.233836384879152e-64

monthly charge vs churn rate
Pearson correlation coefficient: 0.19335642223784708
P-value: 2.706645606888192e-60

Correlation of Churn and MonthlyCharges:
                MonthlyCharges  Churn_numeric
MonthlyCharges        1.000000       0.193356
Churn_numeric         0.193356       1.000000
Correlation of Churn and TotalCharges:
               TotalCharges  Churn_numeric
TotalCharges       1.000000      -0.199428
Churn_numeric     -0.199428       1.000000

calculated churn rate : 26.536987079369588  

churn rate percentage is less but can't be ignored for the growth


Skill used : Data visualization, statistical analysis
Exploratory data analysis
Understanding of customer demographic
variables
Churn rate calculation.

Task 3 [Customer Segmentation]: performed different operation to gain insite of customer segmentation what are  percised relation of other factor with churn how they are affecting it
operation done : elbow method and k-means

elbow method is ploted how much clusters are formed in the data set , the plot is between wcss(within cluster sum of square) and tenure , wcss ws total charges ,  wcss vs contracts ,

then k-means scatter plot are ploted to uninformed model is given and ploted to know what type of customer show churn , based on different based on monthly charges  , total charges and contrant

At tenure < 500 churn rate is high
At monthly charges < 500 churn rate is high


skills used : Segmentation techniques
Understanding of customer behavior
Churn analysis within segments
Identifying high-value customers.

Task 4 [Churn Prediction Model]: made the model that fits the data set well spilt the data set in 20% for testing and 80% for training purpose used 'Logistic Regression','Decision Tree','Random Forest' for regression and ploted  multiple models using metrics like accuracy, precision,recall, and F1-score. Perform feature selection
and hyperparameter tuning are used to check the model is correct hoew much it is deviated from the actual result  to get precise result dynamically

 problem faced :"can't able to solve the error "

 skills used : Machine learning algorithms
Model training and evaluation,
Feature selection, hyperparameter tuning
Understanding of classification metrics.